In [2]:
import os
import xarray as xr
import pandas as pd

import xarray as xr
import matplotlib.pyplot as plt
import numpy as np
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# Open the GEBCO NetCDF file
file_path = "/Users/lb962/Documents/Phd/AExtremesEOF/data/GEBCO/gebco_2024_n70.0_s47.0_w-20.0_e15.0.nc"
GEBCO_ds = xr.open_dataset(file_path)

#ds = xr.open_mfdataset(nc_files, combine='by_coords')  # or combine='nested' with concat_dim='time' if needed
ds_sealevel = xr.open_dataset("/Users/lb962/Documents/GitHub/ESL/data/processed/filtered_GESLA4_sea_level.nc")
# extract station ids and their lat/lon
stations = ds_sealevel['station'].values
lats = ds_sealevel['latitude'].values
lons = ds_sealevel['longitude'].values

# build mapping from (lat, lon) to station index
latlon_to_station = {
    (float(lat), float(lon)): int(stn)
    for stn, lat, lon in zip(stations, lats, lons)
}

# if you want unique lat/lon pairs:
unique_latlon = list(latlon_to_station.keys())

# Open ERA5 datasets

In [3]:
import zipfile
import os

# Source folder where zip files are stored
source_folder = "/Users/lb962/Documents/GitHub/ESL/data/ERA5_by_location"

# Destination folder where everything will be unzipped
dest_folder = os.path.join(source_folder, "unzipped")
os.makedirs(dest_folder, exist_ok=True)

for file in os.listdir(source_folder):
    if file.lower().endswith(".zip"):  # only process .zip files
        file_path = os.path.join(source_folder, file)

        extract_dir = os.path.join(dest_folder, os.path.splitext(file)[0])
        os.makedirs(extract_dir, exist_ok=True)

        try:
            with zipfile.ZipFile(file_path, 'r') as zip_ref:
                zip_ref.extractall(extract_dir)
        except zipfile.BadZipFile:
            print(f"⚠️ Skipping bad zip file: {file}")
    else:
        print(f"Skipping non-zip file: {file}")


Skipping non-zip file: unzipped
Skipping non-zip file: .DS_Store
⚠️ Skipping bad zip file: ERA5_55.00_-8.50.zip


In [3]:
import os
import glob
import xarray as xr
import numpy as np

unzipped_root = "/Users/lb962/Documents/GitHub/ESL/data/ERA5_by_location/unzipped"

nc_files = sorted(glob.glob(os.path.join(unzipped_root, "**", "*.nc"), recursive=True))
if not nc_files:
    raise FileNotFoundError(f"No .nc files found under: {unzipped_root}")
print(f"Found {len(nc_files)} NetCDF files")

def promote_latlon_to_dims(ds: xr.Dataset) -> xr.Dataset:
    """Ensure latitude/longitude are dims (length-1) so open_mfdataset can concat by coords."""
    # normalize coordinate names
    rename_map = {}
    if "lat" in ds and "latitude" not in ds:   rename_map["lat"] = "latitude"
    if "lon" in ds and "longitude" not in ds: rename_map["lon"] = "longitude"
    if rename_map:
        ds = ds.rename(rename_map)

    # make sure they are coords (not just data variables)
    for c in ("latitude", "longitude"):
        if c in ds and c not in ds.coords:
            ds = ds.set_coords(c)

    # promote scalars to 1-length dims
    for c in ("latitude", "longitude"):
        if c in ds:
            if ds[c].ndim == 0:
                # scalar -> 1D with itself as the sole value
                val = ds[c].values.item() if np.ndim(ds[c].values) == 0 else ds[c].values
                ds = ds.expand_dims({c: [val]})
            elif ds[c].ndim == 1:
                # already a dim-sized coord; ensure it's a dimension
                if c not in ds.dims:
                    # if 1D coord but not a dimension, try to set index
                    ds = ds.set_index({c: ds[c]})
            else:
                # if 2D lat/lon grids show up, leave as-is (not a point dataset)
                pass

    return ds

ds = xr.open_mfdataset(
    nc_files,
    engine="netcdf4",
    combine="by_coords",
    preprocess=promote_latlon_to_dims,  # <- key step
    parallel=True,
    data_vars="minimal",
    coords="minimal",
    compat="no_conflicts",
    combine_attrs="drop",
)

Found 226 NetCDF files


/Users/lb962/miniconda3/envs/telec_env/lib/python3.10/site-packages/dask/array/core.py:5097: PerformanceWarning: Increasing number of chunks by factor of 48
  result = blockwise(
/Users/lb962/miniconda3/envs/telec_env/lib/python3.10/site-packages/dask/array/core.py:5097: PerformanceWarning: Increasing number of chunks by factor of 48
  result = blockwise(
/Users/lb962/miniconda3/envs/telec_env/lib/python3.10/site-packages/dask/array/core.py:5097: PerformanceWarning: Increasing number of chunks by factor of 48
  result = blockwise(
/Users/lb962/miniconda3/envs/telec_env/lib/python3.10/site-packages/dask/array/core.py:5097: PerformanceWarning: Increasing number of chunks by factor of 48
  result = blockwise(
/Users/lb962/miniconda3/envs/telec_env/lib/python3.10/site-packages/dask/array/core.py:5097: PerformanceWarning: Increasing number of chunks by factor of 48
  result = blockwise(
/Users/lb962/miniconda3/envs/telec_env/lib/python3.10/site-packages/dask/array/core.py:5097: PerformanceW

# combine all

In [4]:
import glob
import re
import pandas as pd
import xarray as xr
from pathlib import Path
import math

# --- inputs you must already have ---
root = Path("/Users/lb962/Documents/GitHub/ESL/data/ERA5_by_location/unzipped")
# unique_latlon = [...]                    # your list
# ds_sealevel = xr.open_dataset("...")     # opened earlier
# latlon_to_station = {...}                # built earlier

# haversine
R = 6371.0088
def dist(a, b):
    (lat1, lon1), (lat2, lon2) = a, b
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1); dl = math.radians(lon2 - lon1)
    x = math.sin(dphi/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return 2*R*math.atan2(math.sqrt(x), math.sqrt(1-x))

def files_in(folder: Path) -> list[str]:
    nested = sorted(glob.glob(str(folder / "**/*.nc"), recursive=True))
    return nested or sorted(glob.glob(str(folder / "*.nc")))

# --- MISSING PART: populate cands from folder names ERA5_<lat>_<lon> ---
pat = re.compile(r"^ERA5_([+-]?\d+(?:\.\d+)?)_([+-]?\d+(?:\.\d+)?)$")
cands = []
for d in root.iterdir():
    if d.is_dir():
        m = pat.match(d.name)
        if m:
            cands.append((d, float(m.group(1)), float(m.group(2))))
if not cands:
    raise SystemExit(f"No ERA5_<lat>_<lon> folders found in {root}")

opened = []
station_ids = []

for (tlat, tlon) in unique_latlon:
    # nearest folder with files
    ordered = sorted(cands, key=lambda c: dist((tlat, tlon), (c[1], c[2])))
    chosen_folder = chosen_files = None
    for folder, flat_, flon_ in ordered:
        f = files_in(folder)
        if f:
            chosen_folder, chosen_files = folder, f
            break
    if chosen_folder is None:
        continue  # or raise

    # station id (exact -> fallback nearest)
    try:
        station_id = int(latlon_to_station[(float(tlat), float(tlon))])
    except KeyError:
        sl_lats = ds_sealevel["latitude"].values
        sl_lons = ds_sealevel["longitude"].values
        dists = (sl_lats - tlat)**2 + (sl_lons - tlon)**2
        station_id = int(ds_sealevel["station"].values[dists.argmin()])

    ds_loc = xr.open_mfdataset(chosen_files, combine="by_coords", compat="override", parallel=True)

    # reduce to one gridpoint & drop lat/lon
    if "latitude" in ds_loc.dims and "longitude" in ds_loc.dims:
        ds_loc = ds_loc.sel(latitude=float(tlat), longitude=float(tlon), method="nearest")
    for name in ("latitude", "longitude", "lat", "lon"):
        if name in ds_loc.dims:
            ds_loc = ds_loc.isel({name: 0}, drop=True)
        if name in ds_loc.coords:
            ds_loc = ds_loc.reset_coords(name, drop=True)
        if name in ds_loc.variables:
            ds_loc = ds_loc.drop_vars(name, errors="ignore")

    # attach sea level (align time name)
    sl = ds_sealevel.sel(station=station_id)
    sl = sl.rename({"date_time": "valid_time"})
    ds_loc["sea_level"] = sl["sea_level"]

    ds_loc = ds_loc.expand_dims(station=[station_id]).assign_coords(
        station=("station", [station_id]),
        station_latitude=("station", [float(tlat)]),
        station_longitude=("station", [float(tlon)]),
    )

    opened.append(ds_loc)
    station_ids.append(station_id)


In [5]:

# --- Concatenate along station (avoid coordinate merge ambiguity) ---
if not opened:
    raise RuntimeError("No station datasets were opened.")

combined = xr.concat(
    opened,
    dim="station",
    data_vars="all",
    coords="minimal",        # don't try to merge per-dataset coords
    compat="override",
    combine_attrs="override",
    join="outer",            # allow different time coverages
)


In [ ]:
#select = combined.sel(station=[1,2,3,4,5,6,7,8,10])

In [ ]:
#select = combined.isel(station=slice(10, 20))

In [6]:
select = combined.isel(station=slice(21, 30))

In [18]:
#select = combined.isel(station=slice(10, 20))

In [ ]:
import numpy as np
import xarray as xr

# --- robust per-station detider (always returns arrays; NaNs on any issue) ---
def _detide_1d(t, y, lat, min_points=48):
    # t: datetime64[ns] 1D, y: float 1D, lat: scalar
    from utide import solve, reconstruct

    # Default outputs (all NaNs), in case anything fails
    tide  = np.full_like(y, np.nan, dtype=float)
    resid = np.full_like(y, np.nan, dtype=float)

    # Require enough valid points
    mask = ~np.isnan(y)
    if np.count_nonzero(mask) < min_points:
        return tide, resid

    t_clean = t[mask]
    y_clean = y[mask]

    try:
        coef = solve(t_clean, y_clean, lat=lat, method='ols', conf_int='linear')
        reconstructed = reconstruct(t, coef)  # UTide interpolates to full t
        tide = reconstructed.h.astype(float)
        resid = (y - tide).astype(float)
    except Exception:
        # Leave NaNs if anything goes wrong for this station
        pass

    return tide, resid


# ---- batched processing and saving ----
total_stations = combined.sizes["station"]

batch_size = 10
start_idx = 0

for start in range(start_idx, total_stations, batch_size):
    end = min(start + batch_size, total_stations)  # exclusive

    print(f"Processing stations {start}:{end} ...")

    select = combined.isel(station=slice(start, end))

    t_da   = select["valid_time"]              # (time)
    y_da   = select["sea_level"].values        # (station, time)
    lat_da = select["station_latitude"]        # (station,)

    tide_da, resid_da = xr.apply_ufunc(
        _detide_1d,
        t_da, y_da, lat_da,
        input_core_dims=[["valid_time"], ["valid_time"], []],
        output_core_dims=[["valid_time"], ["valid_time"]],
        vectorize=True,
        dask="parallelized",
        output_dtypes=[float, float],
    )

    # Attach results and save this batch
    select = select.assign(
        tide=(("station", "valid_time"), tide_da.data),
        residual=(("station", "valid_time"), resid_da.data),
    )

    out_path = f"/Users/lb962/Documents/GitHub/ESL/data/ml_ready4/small_ds{start}{end}.nc"
    select.to_netcdf(out_path)
    print(f"Saved: {out_path}")


Processing stations 80:90 ...
solve: matrix prep ... solution ... done.
prep/calcs ... done.
solve: matrix prep ... solution ... done.
prep/calcs ... done.
solve: matrix prep ... solution ... done.
prep/calcs ... done.
solve: matrix prep ... solution ... done.
prep/calcs ... done.
solve: matrix prep ... solution ... done.
prep/calcs ... done.
solve: matrix prep ... solution ... done.
prep/calcs ... done.
solve: matrix prep ... solution ... done.
prep/calcs ... done.
solve: matrix prep ... solution ... done.
prep/calcs ... done.
solve: matrix prep ... solution ... done.
prep/calcs ... done.
Saved: /Users/lb962/Documents/GitHub/ESL/data/ml_ready4/small_ds8090.nc
Processing stations 90:100 ...
solve: matrix prep ... solution ... done.
prep/calcs ... done.
solve: matrix prep ... solution ... done.
prep/calcs ... done.
solve: matrix prep ... solution ... done.
prep/calcs ... done.
solve: matrix prep ... solution ... done.
prep/calcs ... done.
solve: matrix prep ... solution ... done.
prep/c

### DONE